# E-Wallet Data & AI Platform — final coursework demo

This read-only notebook walks through the persisted lakehouse through Trino. It
checks local endpoints, inventories Bronze/Silver/Gold, reads current row
counts, runs representative analytics and quality queries, displays selected
contracts, and optionally checks DataHub.

It does **not** generate data, run Spark transformations, register tables,
change metadata, or publish assertions.

## 1. Configuration and helpers

Run the notebook from the repository root when possible. Local defaults match
the documented stack and can be overridden with environment variables:
`TRINO_HOST`, `TRINO_PORT`, `MINIO_URL`, and `DATAHUB_GMS_URL`.

The Trino helper uses the HTTP statement API and Python's standard library, so
no additional Trino client is required by the notebook.

In [ ]:
import json
import os
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

TRINO_HOST = os.getenv("TRINO_HOST", "localhost")
TRINO_PORT = int(os.getenv("TRINO_PORT", "8081"))
TRINO_URL = f"http://{TRINO_HOST}:{TRINO_PORT}"
MINIO_URL = os.getenv("MINIO_URL", "http://localhost:9000")
DATAHUB_GMS_URL = os.getenv("DATAHUB_GMS_URL", "http://localhost:8080")


def find_repo_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "data_platform" / "contracts").is_dir() and (candidate / "README.md").is_file():
            return candidate
    raise RuntimeError("Repository root not found. Start Jupyter inside the cloned repository.")


REPO_ROOT = find_repo_root()
print(f"Repository root: {REPO_ROOT}")
print(f"Trino:           {TRINO_URL}")
print(f"MinIO:           {MINIO_URL}")
print(f"DataHub GMS:     {DATAHUB_GMS_URL} (optional)")


def fetch_json(url: str, timeout: int = 5):
    request = Request(url, headers={"Accept": "application/json"})
    with urlopen(request, timeout=timeout) as response:
        body = response.read().decode("utf-8")
        return response.status, json.loads(body) if body else None


def probe(url: str, timeout: int = 5):
    try:
        request = Request(url)
        with urlopen(request, timeout=timeout) as response:
            response.read(1)
            return True, f"HTTP {response.status}"
    except (HTTPError, URLError, TimeoutError, OSError) as exc:
        return False, str(exc)


def trino_query(sql: str):
    request = Request(
        f"{TRINO_URL}/v1/statement",
        data=sql.encode("utf-8"),
        method="POST",
        headers={
            "X-Trino-User": "coursework-demo",
            "X-Trino-Catalog": "delta",
            "Content-Type": "text/plain; charset=utf-8",
        },
    )
    columns = []
    rows = []

    while True:
        try:
            with urlopen(request, timeout=60) as response:
                payload = json.loads(response.read().decode("utf-8"))
        except (HTTPError, URLError, TimeoutError, OSError) as exc:
            raise RuntimeError(f"Trino request failed at {request.full_url}: {exc}") from exc

        if payload.get("error"):
            error = payload["error"]
            raise RuntimeError(
                f"Trino query failed: {error.get('errorName', 'UNKNOWN')} — "
                f"{error.get('message', error)}"
            )
        if payload.get("columns"):
            columns = [column["name"] for column in payload["columns"]]
        rows.extend(payload.get("data", []))

        next_uri = payload.get("nextUri")
        if not next_uri:
            break
        request = Request(next_uri, headers={"X-Trino-User": "coursework-demo"})

    return columns, rows


def print_result(columns, rows, max_rows=20):
    shown = rows[:max_rows]
    if not columns:
        print("(query returned no tabular result)")
        return
    string_rows = [["NULL" if value is None else str(value) for value in row] for row in shown]
    widths = []
    for index, name in enumerate(columns):
        values = [len(row[index]) for row in string_rows] if string_rows else [0]
        widths.append(min(max(len(name), *values), 48))
    line = " | ".join(name.ljust(widths[i]) for i, name in enumerate(columns))
    print(line)
    print("-+-".join("-" * width for width in widths))
    for row in string_rows:
        print(" | ".join(row[i][:widths[i]].ljust(widths[i]) for i in range(len(columns))))
    if len(rows) > max_rows:
        print(f"... {len(rows) - max_rows} more rows")

## 2. Runtime health

Trino is required for the remaining data cells. MinIO confirms the backing
object store. DataHub is optional because it is a separate Quickstart runtime.

In [ ]:
checks = [
    ("Trino", f"{TRINO_URL}/v1/info", True),
    ("MinIO", f"{MINIO_URL.rstrip('/')}/minio/health/ready", True),
    ("DataHub GMS", f"{DATAHUB_GMS_URL.rstrip('/')}/config", False),
]

for name, url, required in checks:
    ok, detail = probe(url)
    requirement = "required" if required else "optional"
    state = "PASS" if ok else "UNAVAILABLE"
    print(f"{state:11} | {name:12} | {requirement:8} | {detail}")

## 3. Lakehouse inventory

These commands prove that the schemas and registered tables are visible through
the current Trino metastore.

In [ ]:
inventory_queries = [
    ("Schemas", "SHOW SCHEMAS FROM delta"),
    ("Bronze tables", "SHOW TABLES FROM delta.bronze_zone"),
    ("Silver tables", "SHOW TABLES FROM delta.silver_zone"),
    ("Gold tables", "SHOW TABLES FROM delta.gold_zone"),
]

for title, sql in inventory_queries:
    print(f"\n{title}")
    columns, rows = trino_query(sql)
    print_result(columns, rows, max_rows=30)

## 4. Current layer row counts

The notebook reads counts from the persisted tables. It does not assume a fixed
dataset size.

In [ ]:
row_count_sql = """
SELECT layer, table_name, row_count
FROM (
    SELECT 'bronze' AS layer, 'transactions' AS table_name,
           count(*) AS row_count FROM delta.bronze_zone.transactions
    UNION ALL
    SELECT 'silver', 'transactions', count(*) FROM delta.silver_zone.transactions
    UNION ALL
    SELECT 'gold', 'fact_transactions', count(*) FROM delta.gold_zone.fact_transactions
    UNION ALL
    SELECT 'gold', 'obt_transaction_enriched', count(*) FROM delta.gold_zone.obt_transaction_enriched
    UNION ALL
    SELECT 'gold', 'feat_user_90d', count(*) FROM delta.gold_zone.feat_user_90d
)
ORDER BY layer, table_name
"""
columns, rows = trino_query(row_count_sql)
print_result(columns, rows)

## 5. Representative analytics

The examples use fields produced by the current Gold builders: transaction
status/type, the merchant-performance aggregate, and 90-day user features.

In [ ]:
analytics = [
    (
        "Transactions by status and type",
        """
        SELECT status, type, count(*) AS transaction_count,
               round(sum(amount), 2) AS total_amount
        FROM delta.gold_zone.fact_transactions
        GROUP BY status, type
        ORDER BY transaction_count DESC
        """,
    ),
    (
        "Top merchants by total amount",
        """
        SELECT merchant_name, category, transaction_count,
               round(total_amount, 2) AS total_amount,
               distinct_users
        FROM delta.gold_zone.opt_merchant_performance
        ORDER BY total_amount DESC
        LIMIT 10
        """,
    ),
    (
        "Sample 90-day user features",
        """
        SELECT user_id,
               f_user_total_transactions_90d,
               round(f_user_avg_transaction_amount_90d, 2) AS avg_amount_90d,
               round(f_user_failed_transaction_rate_90d, 4) AS failed_rate_90d,
               f_user_distinct_merchants_90d
        FROM delta.gold_zone.feat_user_90d
        ORDER BY f_user_total_transactions_90d DESC
        LIMIT 10
        """,
    ),
]

for title, sql in analytics:
    print(f"\n{title}")
    columns, rows = trino_query(sql)
    print_result(columns, rows)

## 6. Selected read-only data-quality evidence

These are compact demonstrations of the same categories enforced by the full
validators: required values, uniqueness, population equality, and ranges. Run
the validator modules for the complete fail-hard rule sets.

In [ ]:
quality_queries = [
    (
        "Silver required-field null rows",
        """
        SELECT count_if(
            transaction_id IS NULL OR account_id IS NULL OR user_id IS NULL OR
            device_id IS NULL OR type IS NULL OR amount IS NULL OR
            currency IS NULL OR status IS NULL OR channel IS NULL OR
            old_balance IS NULL OR new_balance IS NULL OR
            timestamp IS NULL OR ingested_at IS NULL OR event_date IS NULL
        ) AS invalid_rows
        FROM delta.silver_zone.transactions
        """,
    ),
    (
        "Silver duplicate transaction ID groups",
        """
        SELECT count(*) AS duplicate_groups
        FROM (
            SELECT transaction_id
            FROM delta.silver_zone.transactions
            GROUP BY transaction_id
            HAVING count(*) > 1
        )
        """,
    ),
    (
        "Gold population relationships",
        """
        SELECT
            (SELECT count(*) FROM delta.silver_zone.transactions) AS silver_transactions,
            (SELECT count(*) FROM delta.gold_zone.fact_transactions) AS fact_transactions,
            (SELECT count(*) FROM delta.gold_zone.obt_transaction_enriched) AS obt_transactions,
            (SELECT count(*) FROM delta.gold_zone.dim_user) AS dim_users,
            (SELECT count(*) FROM delta.gold_zone.feat_user_90d) AS feature_users
        """,
    ),
    (
        "Feature failed-rate range",
        """
        SELECT min(f_user_failed_transaction_rate_90d) AS min_failed_rate,
               max(f_user_failed_transaction_rate_90d) AS max_failed_rate,
               count_if(
                   f_user_failed_transaction_rate_90d < 0 OR
                   f_user_failed_transaction_rate_90d > 1
               ) AS out_of_range_rows
        FROM delta.gold_zone.feat_user_90d
        """,
    ),
]

for title, sql in quality_queries:
    print(f"\n{title}")
    columns, rows = trino_query(sql)
    print_result(columns, rows)

## 7. Data contracts

The contracts record grain, logical key, columns, and quality relationships for
four selected interfaces. This cell reads them; it does not validate or modify
them.

In [ ]:
try:
    import yaml
except ImportError:
    yaml = None

contract_paths = [
    REPO_ROOT / "data_platform" / "contracts" / "silver_transactions.yml",
    REPO_ROOT / "data_platform" / "contracts" / "gold_fact_transactions.yml",
    REPO_ROOT / "data_platform" / "contracts" / "gold_obt_transaction_enriched.yml",
    REPO_ROOT / "data_platform" / "contracts" / "gold_feat_user_90d.yml",
]

for path in contract_paths:
    print(f"\n{path.relative_to(REPO_ROOT)}")
    if yaml is None:
        print("PyYAML is unavailable; showing the first 20 source lines.")
        print("".join(path.read_text(encoding="utf-8").splitlines(keepends=True)[:20]))
        continue
    contract = yaml.safe_load(path.read_text(encoding="utf-8"))
    required_columns = [
        column["name"] for column in contract.get("columns", [])
        if not column.get("nullable", True)
    ]
    print(f"dataset:          {contract['dataset']}")
    print(f"contract_version: {contract['contract_version']}")
    print(f"grain:            {contract['grain']}")
    print(f"logical_key:      {contract['logical_key']}")
    print(f"required_columns: {required_columns}")
    print(f"quality_rules:    {len(contract.get('quality_rules', []))}")

## 8. Optional DataHub status

When GMS is running, this reads its public configuration and reports the server
version. Metadata ingestion, lineage publication, and assertion publication are
separate explicit commands and are not run by this notebook.

In [ ]:
datahub_config_url = f"{DATAHUB_GMS_URL.rstrip('/')}/config"
try:
    status, config = fetch_json(datahub_config_url)
    print(f"PASS | DataHub GMS | HTTP {status}")
    versions = config.get("versions", {}) if isinstance(config, dict) else {}
    print(f"server version: {versions.get('acryldata/datahub', versions or 'not reported')}")
    print("No metadata or assertion write was performed.")
except (HTTPError, URLError, TimeoutError, OSError, json.JSONDecodeError) as exc:
    print(f"OPTIONAL | DataHub GMS unavailable at {datahub_config_url} | {exc}")
    print("The Trino lakehouse demo is still valid without the separate DataHub runtime.")

## 9. Conclusion

This walkthrough demonstrated the current persisted lakehouse through its SQL
query surface, sampled analytics, checked representative quality invariants,
and linked those invariants to repository-owned contracts and optional DataHub
metadata. The full validators remain the authoritative executable gates, and
the PyFlink/CDC paths remain experimental until they gain persistent sinks.